# Merrick Bank Customer Retention Agent (RAG + LangChain + Gradio)

**Course:** UTD AI/ML Bootcamp, Agentic AI capstone session
**Author:** Alessandra Rye

Merrick Bank already has churn and sentiment models that flag at-risk customers, but nothing that turns those flags into a conversation. This notebook builds a retrieval-augmented retention assistant that:

1. Loads the bank's unstructured documents (FAQs, policies, product summaries), chunks them, embeds them with a Hugging Face sentence transformer, and stores them in a FAISS index.
2. Loads the structured churn data (plus sentiment labels) into a `CustomerDB` class with a `get_context(customer_id)` lookup.
3. Combines the retriever, the customer lookup, and an LLM into one LangChain `Runnable` chain with a system prompt that carries the retrieved context, the customer profile, and compliance rules.
4. Wraps the chain in a Gradio chat UI that shows citations, keeps chat history, and handles missing context without making things up.
5. Tests the three sample queries from the assignment and scores grounding, personalization, tone, and coherence.

| Task | Section |
|---|---|
| 1. Data preparation | Section 1 |
| 2. Document ingestion and embedding | Section 2 |
| 3. Customer data integration | Section 3 |
| 4. Building the RAG chain | Section 4 |
| 5. Gradio conversational interface | Section 5 |
| 6. Testing and evaluation | Section 6 |

**Before running:** install the packages in `requirements.txt` and either put `OPENAI_API_KEY` in a `.env` file next to this notebook, or have Ollama running locally (see the README).

## 0. Setup and configuration

In [1]:
import json
import os
import re
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()  # picks up OPENAI_API_KEY, LLM_PROVIDER, etc. from a local .env file

# Project folders (Task 1.4). Works both as a notebook and as app.py.
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_DIR = BASE_DIR / "data"
DOCS_DIR = BASE_DIR / "docs"
INDEX_DIR = BASE_DIR / "index"

CHURN_CSV = DATA_DIR / "Churn_Modeling1.csv"
SENTIMENT_CSV = DATA_DIR / "Churn_With_Sentiment.csv"

# Retrieval settings
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 75
TOP_K = 4
MIN_SIMILARITY = 0.25  # cosine similarity floor; chunks below this are treated as "not relevant"

# LLM settings. LLM_PROVIDER can be "openai" or "ollama".
# If it is not set, OpenAI is used when a key is present, otherwise Ollama.
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")
MAX_HISTORY_TURNS = 6  # how many past question/answer pairs are sent back to the LLM

## 1. Task 1: Data preparation

The structured data is two CSV files in `data/`:

- `Churn_Modeling1.csv`: 10,000 customers with profile fields and the `Exited` churn flag.
- `Churn_With_Sentiment.csv`: the same customers with one extra column, `Sentiment_Label`.

The unstructured data is six documents in `docs/` (one FAQ in Markdown and five policy/product text files). This section creates the project folders, checks the CSVs, and verifies that every document can be read as UTF-8.

In [2]:
def prepare_folders():
    """Task 1.4: make sure data/, docs/, and index/ exist."""
    for folder in (DATA_DIR, DOCS_DIR, INDEX_DIR):
        folder.mkdir(parents=True, exist_ok=True)
    return {f.name: f.exists() for f in (DATA_DIR, DOCS_DIR, INDEX_DIR)}


def list_doc_files():
    """All .txt and .md files under docs/, skipping Jupyter checkpoint copies."""
    files = [p for ext in ("*.txt", "*.md") for p in DOCS_DIR.rglob(ext)]
    return sorted(p for p in files if ".ipynb_checkpoints" not in p.parts)


def verify_documents():
    """Task 1.3: confirm each document decodes as UTF-8 and is not empty."""
    rows = []
    for path in list_doc_files():
        raw = path.read_bytes()
        try:
            text = raw.decode("utf-8")
            readable, encoding = True, "utf-8"
        except UnicodeDecodeError:
            text = raw.decode("latin-1")
            readable, encoding = False, "NOT utf-8 (fell back to latin-1)"
        rows.append({
            "file": path.name,
            "bytes": len(raw),
            "characters": len(text),
            "lines": text.count("\n") + 1,
            "encoding": encoding,
            "windows_line_endings": "\r\n" in text,
            "readable": readable and len(text.strip()) > 0,
        })
    return pd.DataFrame(rows)

In [3]:
print("Folders:", prepare_folders())

churn_raw = pd.read_csv(CHURN_CSV)
sentiment_raw = pd.read_csv(SENTIMENT_CSV)

print(f"\nChurn data:      {churn_raw.shape[0]:,} rows x {churn_raw.shape[1]} columns")
print(f"Sentiment data:  {sentiment_raw.shape[0]:,} rows x {sentiment_raw.shape[1]} columns")
print("Missing values:  churn =", int(churn_raw.isna().sum().sum()),
      "| sentiment =", int(sentiment_raw.isna().sum().sum()))
print("CustomerId unique in both files:", churn_raw.CustomerId.is_unique and sentiment_raw.CustomerId.is_unique)
print("Same customers in both files:  ", set(churn_raw.CustomerId) == set(sentiment_raw.CustomerId))
print(f"Churn rate (Exited = 1):        {churn_raw.Exited.mean():.1%}")
churn_raw.head()

Folders: {'data': True, 'docs': True, 'index': True}

Churn data:      10,000 rows x 13 columns
Sentiment data:  10,000 rows x 14 columns
Missing values:  churn = 0 | sentiment = 0
CustomerId unique in both files: True
Same customers in both files:   True
Churn rate (Exited = 1):        20.4%


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# Sentiment labels and how they line up with churn
display(sentiment_raw.Sentiment_Label.value_counts().to_frame("customers"))
pd.crosstab(sentiment_raw.Sentiment_Label, sentiment_raw.Exited, normalize="index").round(3)

,customers
Sentiment_Label,
Negative,8441
Positive,1349
Neutral,210


Exited,0,1
Sentiment_Label,,
Negative,0.797,0.203
Neutral,0.767,0.233
Positive,0.795,0.205


In [5]:
doc_report = verify_documents()
assert doc_report.readable.all(), "At least one document is unreadable or empty"
doc_report

,file,bytes,characters,lines,encoding,windows_line_endings,readable
0,complaints_handling.txt,1247,1243,36,utf-8,True,True
1,faq_accounts.md,2097,2083,32,utf-8,False,True
2,fee_waiver_policy.txt,1097,1089,27,utf-8,True,True
3,product_summary.txt,1168,1132,38,utf-8,True,True
4,retention_policy.txt,1663,1645,38,utf-8,True,True
5,rewards_and_benefits.txt,1145,1131,31,utf-8,True,True


**What the checks show**

- Both CSVs have 10,000 rows, no missing values, and the same unique `CustomerId` set, so they can be merged one-to-one.
- A few surnames contain a literal `?` (for example `H?`). That is character loss from before the files were handed out, not a loading problem on this end, so it is left as is.
- All six documents decode cleanly as UTF-8. The five `.txt` files use Windows line endings (`\r\n`), which get normalized during loading so they do not affect chunking.
- The zip also carried a `.ipynb_checkpoints` copy of the FAQ. It is excluded so the same FAQ text does not get indexed twice.

## 2. Task 2: Document ingestion and embedding

Pipeline: `DirectoryLoader` + `TextLoader` to load, `RecursiveCharacterTextSplitter` to chunk, `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) to embed, FAISS to store. The index is saved to `index/` so later runs load it instead of re-embedding.

Chunks are 500 characters with 75 characters of overlap. The documents are short numbered lists, so that size keeps each numbered section (for example "3. Retention Offers") mostly intact in one chunk.

In [6]:
def load_documents():
    """Task 2.1: load every .txt and .md file in docs/ with DirectoryLoader + TextLoader."""
    loader = DirectoryLoader(
        str(DOCS_DIR),
        glob=["**/*.txt", "**/*.md"],
        exclude=["**/.ipynb_checkpoints/**"],
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    docs = loader.load()
    for doc in docs:
        doc.page_content = doc.page_content.replace("\r\n", "\n").strip()
        doc.metadata["source"] = Path(doc.metadata["source"]).name  # keep just the file name for citations
    return sorted(docs, key=lambda d: d.metadata["source"])


def split_documents(docs):
    """Task 2.2: split documents into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
    return chunks


def get_embeddings():
    """Task 2.3: Hugging Face sentence-transformer embeddings (normalized, so distances map to cosine similarity)."""
    return HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        encode_kwargs={"normalize_embeddings": True},
    )


def build_or_load_index(embeddings, rebuild=False):
    """Task 2.4: build the FAISS index from docs/ and save it to index/, or load the saved copy."""
    index_file = INDEX_DIR / "index.faiss"
    if index_file.exists() and not rebuild:
        # The index was written by this project, so loading its pickle metadata is safe.
        return FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
    chunks = split_documents(load_documents())
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(str(INDEX_DIR))
    return vectorstore

In [7]:
raw_docs = load_documents()
chunks = split_documents(raw_docs)

print(f"Loaded {len(raw_docs)} documents and split them into {len(chunks)} chunks\n")
display(pd.Series([c.metadata["source"] for c in chunks]).value_counts().to_frame("chunks"))

print("Example chunk:\n")
print(chunks[0].metadata)
print(chunks[0].page_content)

Loaded 6 documents and split them into 23 chunks



,chunks
faq_accounts.md,6
retention_policy.txt,5
complaints_handling.txt,3
fee_waiver_policy.txt,3
product_summary.txt,3
rewards_and_benefits.txt,3


Example chunk:

{'source': 'complaints_handling.txt', 'chunk_id': 0}
Complaints Handling — Policy and Process

1. Objective
   Ensure every customer complaint is acknowledged and resolved promptly, maintaining high satisfaction scores.

2. Response Timeline
   - Acknowledgment within 24 hours.
   - Resolution within 5 working days.
   - Complex cases may take up to 10 days.

3. Channels for Complaint
   - Toll-free number
   - Email support
   - Chatbot or live chat
   - Social media escalation


In [8]:
embeddings = get_embeddings()
vectorstore = build_or_load_index(embeddings, rebuild=True)  # rebuild=True so the notebook always reflects docs/

print("Vectors in FAISS index:", vectorstore.index.ntotal)
print("Embedding dimensions:  ", vectorstore.index.d)
print("Saved to:", sorted(p.name for p in INDEX_DIR.iterdir() if not p.name.startswith(".")))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectors in FAISS index: 23
Embedding dimensions:   384
Saved to: ['index.faiss', 'index.pkl']


### Retrieval with a relevance floor

Plain top-k retrieval always returns something, even for a question the documents cannot answer. To support the "handle missing context gracefully" requirement, `retrieve()` converts FAISS distances to cosine similarity and drops chunks below `MIN_SIMILARITY`. An off-topic question then comes back with zero chunks, and the chain can say so instead of forcing an answer out of unrelated text.

In [9]:
def retrieve(vectorstore, query, k=TOP_K, min_similarity=MIN_SIMILARITY):
    """Top-k chunks for a query, filtered by cosine similarity.

    FAISS returns squared L2 distance. With normalized embeddings,
    cosine similarity = 1 - distance / 2.
    """
    results = []
    for doc, distance in vectorstore.similarity_search_with_score(query, k=k):
        similarity = 1 - float(distance) / 2
        if similarity >= min_similarity:
            # Return a copy so the score from this query is not overwritten by a later query that hits the same chunk.
            results.append(Document(page_content=doc.page_content,
                                    metadata={**doc.metadata, "similarity": round(similarity, 3)}))
    return results


def format_docs(docs):
    """Number the retrieved chunks so the LLM can cite them as [1], [2], ..."""
    if not docs:
        return "NO RELEVANT DOCUMENTS WERE FOUND."
    return "\n\n".join(
        f"[{i}] (source: {d.metadata['source']})\n{d.page_content}" for i, d in enumerate(docs, start=1)
    )

In [10]:
# Top-4 similarities with the floor switched off, so the MIN_SIMILARITY setting can be checked against real scores.
# On-topic questions should score clearly above the floor and the off-topic one clearly below it.
probe_questions = [
    "How can I waive my annual fee?",
    "What should we offer to retain customer?",
    "Explain the difference between Gold and Platinum cards.",
    "What is the weather in Columbus today?",
]
for q in probe_questions:
    print(f"Q: {q}")
    for d in retrieve(vectorstore, q, min_similarity=-1):
        kept = "kept   " if d.metadata["similarity"] >= MIN_SIMILARITY else "dropped"
        print(f"   {d.metadata['similarity']:.3f}  {kept}  {d.metadata['source']}")
    print()

Q: How can I waive my annual fee?
   0.658  kept     fee_waiver_policy.txt
   0.506  kept     retention_policy.txt
   0.464  kept     fee_waiver_policy.txt
   0.396  kept     faq_accounts.md

Q: What should we offer to retain customer?
   0.581  kept     retention_policy.txt
   0.478  kept     retention_policy.txt
   0.423  kept     retention_policy.txt
   0.420  kept     retention_policy.txt

Q: Explain the difference between Gold and Platinum cards.
   0.493  kept     product_summary.txt
   0.399  kept     rewards_and_benefits.txt
   0.358  kept     rewards_and_benefits.txt
   0.273  kept     product_summary.txt

Q: What is the weather in Columbus today?
   0.035  dropped  complaints_handling.txt
   0.029  dropped  faq_accounts.md
   -0.001  dropped  complaints_handling.txt
   -0.002  dropped  product_summary.txt



## 3. Task 3: Customer data integration

`CustomerDB` loads the churn file, merges in the sentiment label (the optional step), and exposes `get_context(customer_id)`.

Besides the raw profile, `get_context` adds a few derived fields so the LLM does not have to do policy arithmetic on its own:

- **Retention segment** using the rules in `retention_policy.txt` (high, moderate, or low value based on tenure and credit score).
- **Fee waiver signals** using the criteria in `fee_waiver_policy.txt`.
- **Risk signals** from the churn flag, activity flag, and sentiment label.

The policy's segment rules overlap (a customer can have 2 years of tenure and a credit score under 650). When that happens the customer gets the highest tier they qualify for, which is the reading that favors the customer. Anything the dataset does not contain, such as late payment history, is reported as unknown instead of assumed.

In [11]:
class CustomerDB:
    """Structured customer data: churn profile plus optional sentiment label."""

    def __init__(self, churn_path=CHURN_CSV, sentiment_path=SENTIMENT_CSV):
        df = pd.read_csv(churn_path)
        self.has_sentiment = False
        if sentiment_path and Path(sentiment_path).exists():
            sentiment = pd.read_csv(sentiment_path)[["CustomerId", "Sentiment_Label"]]
            df = df.merge(sentiment, on="CustomerId", how="left")
            self.has_sentiment = True
        self.df = df.set_index("CustomerId")

    def __len__(self):
        return len(self.df)

    @staticmethod
    def _segment(tenure, credit_score):
        """Segmentation rules from retention_policy.txt, highest matching tier wins."""
        if tenure > 3 or credit_score > 750:
            return "High-value"
        if 1 <= tenure <= 3 or 650 <= credit_score <= 750:
            return "Moderate-value"
        return "Low-value"

    @staticmethod
    def _fee_waiver_signals(tenure, credit_score, is_active):
        """Eligibility criteria from fee_waiver_policy.txt, limited to what the data can answer."""
        return {
            "active_12_months_or_more": bool(is_active and tenure >= 1),
            "credit_score_above_700": bool(credit_score > 700),
            "auto_approval_tenure_over_24_months": bool(tenure > 2),
            "late_payment_history": "unknown (not in the dataset)",
            "financial_hardship": "unknown (not in the dataset)",
        }

    def get_context(self, customer_id):
        """Task 3.2: profile, churn details, sentiment, and derived retention fields for one customer.

        Returns None when the ID is not a valid number or is not in the data.
        """
        try:
            customer_id = int(str(customer_id).strip())
        except (TypeError, ValueError):
            return None
        if customer_id not in self.df.index:
            return None

        row = self.df.loc[customer_id]
        tenure, score = int(row["Tenure"]), int(row["CreditScore"])
        exited, active = int(row["Exited"]), int(row["IsActiveMember"])
        sentiment = row["Sentiment_Label"] if self.has_sentiment and pd.notna(row["Sentiment_Label"]) else None

        risk_signals = []
        if exited:
            risk_signals.append("flagged as churned in the churn data (Exited = 1)")
        if not active:
            risk_signals.append("not an active member")
        if sentiment == "Negative":
            risk_signals.append("negative sentiment in recent feedback")

        return {
            "customer_id": customer_id,
            "surname": str(row["Surname"]),
            "geography": str(row["Geography"]),
            "gender": str(row["Gender"]),
            "age": int(row["Age"]),
            "tenure_years": tenure,
            "credit_score": score,
            "balance": float(row["Balance"]),
            "num_products": int(row["NumOfProducts"]),
            "has_credit_card": bool(row["HasCrCard"]),
            "is_active_member": bool(active),
            "estimated_salary": float(row["EstimatedSalary"]),
            "exited": bool(exited),
            "sentiment": sentiment,
            "retention_segment": self._segment(tenure, score),
            "fee_waiver_signals": self._fee_waiver_signals(tenure, score, active),
            "risk_signals": risk_signals,
        }


def resolve_customer_id(customer_id_field, question):
    """Use the ID typed in the UI field, or fall back to an 8-digit ID mentioned in the question."""
    field = str(customer_id_field or "").strip()
    if field:
        return field
    match = re.search(r"\b(\d{8})\b", question or "")
    return match.group(1) if match else None


def format_customer(lookup):
    """Turn a customer lookup result into the text block that goes into the system prompt."""
    if lookup["status"] == "none":
        return "No customer ID was provided. Answer in general terms and do not assume any customer details."
    if lookup["status"] == "not_found":
        return (f"A customer ID was provided ({lookup['id']}) but NO MATCHING RECORD exists in the customer database. "
                "Say that the customer could not be found, ask the user to check the ID, and answer only in general terms.")
    c = lookup["profile"]
    yes_no = lambda flag: "Yes" if flag else "No"
    waiver = c["fee_waiver_signals"]
    return "\n".join([
        f"Customer ID: {c['customer_id']}",
        f"Name on file: {c['surname']} (surname only; no first name is stored)",
        f"Geography: {c['geography']} | Gender: {c['gender']} | Age: {c['age']}",
        f"Tenure: {c['tenure_years']} years | Products held: {c['num_products']} | "
        f"Has credit card: {yes_no(c['has_credit_card'])} | Active member: {yes_no(c['is_active_member'])}",
        f"Credit score: {c['credit_score']} | Balance: {c['balance']:,.2f} | Estimated salary: {c['estimated_salary']:,.2f}",
        f"Churn flag (Exited): {yes_no(c['exited'])}",
        f"Feedback sentiment: {c['sentiment'] or 'not available'}",
        f"Retention segment (derived from the retention policy rules): {c['retention_segment']}",
        "Fee waiver signals (derived from the fee waiver policy criteria): "
        f"active 12+ months = {yes_no(waiver['active_12_months_or_more'])}; "
        f"credit score above 700 = {yes_no(waiver['credit_score_above_700'])}; "
        f"auto-approval (tenure over 24 months) = {yes_no(waiver['auto_approval_tenure_over_24_months'])}; "
        f"late payment history = {waiver['late_payment_history']}; "
        f"financial hardship = {waiver['financial_hardship']}",
        "Risk signals: " + ("; ".join(c["risk_signals"]) if c["risk_signals"] else "none in the data"),
    ])

In [12]:
customer_db = CustomerDB()
print(f"CustomerDB loaded: {len(customer_db):,} customers | sentiment merged: {customer_db.has_sentiment}\n")

ctx = customer_db.get_context(15634602)
print(json.dumps(ctx, indent=2))

CustomerDB loaded: 10,000 customers | sentiment merged: True

{
  "customer_id": 15634602,
  "surname": "Hargrave",
  "geography": "France",
  "gender": "Female",
  "age": 42,
  "tenure_years": 2,
  "credit_score": 619,
  "balance": 0.0,
  "num_products": 1,
  "has_credit_card": true,
  "is_active_member": true,
  "estimated_salary": 101348.88,
  "exited": true,
  "sentiment": "Positive",
  "retention_segment": "Moderate-value",
  "fee_waiver_signals": {
    "active_12_months_or_more": true,
    "credit_score_above_700": false,
    "auto_approval_tenure_over_24_months": false,
    "late_payment_history": "unknown (not in the dataset)",
    "financial_hardship": "unknown (not in the dataset)"
  },
  "risk_signals": [
    "flagged as churned in the churn data (Exited = 1)"
  ]
}


In [13]:
# What the LLM will actually see for this customer, and what happens with a bad ID
print(format_customer({"status": "found", "id": "15634602", "profile": ctx}))
print("\nUnknown ID ->", customer_db.get_context(99999999))
print("Bad input  ->", customer_db.get_context("abc"))

Customer ID: 15634602
Name on file: Hargrave (surname only; no first name is stored)
Geography: France | Gender: Female | Age: 42
Tenure: 2 years | Products held: 1 | Has credit card: Yes | Active member: Yes
Credit score: 619 | Balance: 0.00 | Estimated salary: 101,348.88
Churn flag (Exited): Yes
Feedback sentiment: Positive
Retention segment (derived from the retention policy rules): Moderate-value
Fee waiver signals (derived from the fee waiver policy criteria): active 12+ months = Yes; credit score above 700 = No; auto-approval (tenure over 24 months) = No; late payment history = unknown (not in the dataset); financial hardship = unknown (not in the dataset)
Risk signals: flagged as churned in the churn data (Exited = 1)

Unknown ID -> None
Bad input  -> None


## 4. Task 4: Building the RAG chain

The chain is built from LangChain `Runnable` components:

```
input {question, customer_id, history}
  -> RunnablePassthrough.assign(docs = retriever, customer = CustomerDB lookup)
  -> RunnablePassthrough.assign(context = numbered chunks, customer_block = profile text)
  -> RunnableBranch:
        nothing retrieved, no customer, no history  -> fixed "I can't find that" reply (no LLM call)
        otherwise                                    -> prompt | LLM | StrOutputParser
```

The chain returns the answer together with the retrieved chunks, so the UI can show exactly which sources were used.

The system prompt has three parts: the retrieved context, the customer profile, and compliance guidelines. The compliance rules come straight from the bank's own documents: never promise an offer beyond policy limits, send exceptions to the Retention Desk Manager, acknowledge the concern first, offer one relevant option, and end with a clear action.

In [14]:
SYSTEM_PROMPT = """You are the Merrick Bank Customer Retention Assistant. You help retention agents and \
customers with questions about accounts, cards, fees, rewards, complaints, and retention offers.

COMPLIANCE GUIDELINES (always follow these)
1. Grounding: every statement about a policy, product, fee, rate, timeline, or offer must come from the \
RETRIEVED CONTEXT below. Whenever you use the context, you MUST cite the chunk right after the statement, \
using its number in square brackets, like [1]. Put each number in its own brackets, for example [1][3]. \
An answer that uses the context but has no [n] citation is not acceptable. \
Never invent fees, rates, offers, or eligibility rules.
2. Missing information: if the context does not contain the answer, say plainly that you do not have that \
information in the bank's documents and suggest contacting customer support. Do not guess.
3. Offers: only suggest retention offers that the retention policy lists for this customer's segment. Never \
promise anything beyond policy limits. Any exception must be approved by the Retention Desk Manager.
4. No guarantees: describe eligibility conditions instead of promising approval, unless the policy says the \
request is auto-approved for this customer. If a criterion is marked unknown in the profile, say it still \
has to be verified.
5. Privacy: only discuss the customer in the CUSTOMER PROFILE below. Never reveal or speculate about any \
other customer. Use the profile to personalize, but do not quote the customer's salary or exact balance.
6. Tone: be empathetic, respectful, and transparent. Acknowledge the concern first, use plain language \
without jargon, offer one relevant option where a retention offer applies, and end with a clear next step.
7. Keep amounts and currency symbols exactly as they appear in the context.

CUSTOMER PROFILE
{customer_block}

RETRIEVED CONTEXT
{context}

Keep the answer concise (under about 180 words) unless the user asks for more detail."""

FALLBACK_ANSWER = (
    "I'm sorry, I couldn't find anything in Merrick Bank's policy and product documents that answers that, "
    "so I don't want to guess. I can help with annual fees and fee waivers, card and account products, "
    "rewards, complaints, and retention offers. If you include a customer ID, I can tailor the answer to "
    "that customer. For anything else, please contact customer support."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])


def get_llm():
    """Task 4.3: ChatOpenAI or ChatOllama, chosen by LLM_PROVIDER or by whether an OpenAI key is present."""
    provider = os.getenv("LLM_PROVIDER") or ("openai" if os.getenv("OPENAI_API_KEY") else "ollama")
    if provider.lower() == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=OPENAI_MODEL, temperature=0.2)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=0.2)


def build_chain(vectorstore, customer_db, llm):
    """Task 4.1: retriever + customer lookup + prompt + LLM, composed from Runnables."""

    def lookup_customer(x):
        cid = resolve_customer_id(x.get("customer_id"), x["question"])
        if cid is None:
            return {"status": "none", "id": None, "profile": None}
        profile = customer_db.get_context(cid)
        return {"status": "found" if profile else "not_found", "id": cid, "profile": profile}

    gather = RunnablePassthrough.assign(
        # The customer ID is stripped before searching: digits add noise to the embedding and match nothing in the docs.
        docs=RunnableLambda(lambda x: retrieve(vectorstore, re.sub(r"\b\d{8}\b", "", x["question"]).strip())),
        customer=RunnableLambda(lookup_customer),
    )
    shape = RunnablePassthrough.assign(
        context=lambda x: format_docs(x["docs"]),
        customer_block=lambda x: format_customer(x["customer"]),
        history=lambda x: x.get("history", []),
    )
    generate = prompt | llm | StrOutputParser()

    nothing_to_go_on = lambda x: not x["docs"] and x["customer"]["status"] == "none" and not x["history"]
    answer = RunnableBranch(
        (nothing_to_go_on, RunnableLambda(lambda x: FALLBACK_ANSWER)),
        generate,
    )
    return gather | shape | RunnablePassthrough.assign(answer=answer)


def to_messages(turns):
    """Convert [(question, answer), ...] into LangChain messages, keeping only the most recent turns."""
    messages = []
    for question, answer in turns[-MAX_HISTORY_TURNS:]:
        messages += [HumanMessage(content=question), AIMessage(content=answer)]
    return messages


def ask(chain, question, customer_id="", turns=None):
    """Run the chain once and return the full result (answer, docs, customer lookup)."""
    return chain.invoke({
        "question": question.strip(),
        "customer_id": customer_id,
        "history": to_messages(turns or []),
    })


def format_citations(docs):
    """Markdown list of the sources behind an answer, numbered to match the [n] citations."""
    if not docs:
        return "_No policy documents were retrieved for this answer._"
    lines = []
    for i, d in enumerate(docs, start=1):
        snippet = " ".join(d.page_content.split())[:160]
        lines.append(f"**[{i}] {d.metadata['source']}** (similarity {d.metadata.get('similarity', 0):.2f}): {snippet}...")
    return "\n\n".join(lines)

In [15]:
llm = get_llm()
chain = build_chain(vectorstore, customer_db, llm)
print("LLM:", type(llm).__name__, getattr(llm, "model_name", None) or getattr(llm, "model", ""))

from IPython.display import Markdown, display

result = ask(chain, "How can I waive my annual fee?")
display(Markdown(result["answer"]))
display(Markdown(format_citations(result["docs"])))

LLM: ChatOpenAI gpt-4o-mini


To waive your annual fee, you need to meet certain eligibility criteria. You must be an active customer for at least 12 months, have no late payments in the last 3 cycles, and either have a credit score above 700 or be experiencing verified financial hardship [1]. 

If you qualify, you can request a one-time annual fee waiver, which is applicable once per card every 12 months [1]. 

If you believe you meet these conditions, you can reach out to customer support to initiate the waiver request. They will guide you through the process.

**[1] fee_waiver_policy.txt** (similarity 0.66): Fee Waiver Policy 1. Eligibility - Active customer for at least 12 months. - No late payment in the last 3 cycles. - Credit score >700 or verified financial har...

**[2] retention_policy.txt** (similarity 0.51): 3. Retention Offers - High-value: fee waiver up to ₹1,000, bonus reward points, or interest reduction of 0.5%. - Moderate-value: 50% annual fee waiver or EMI in...

**[3] fee_waiver_policy.txt** (similarity 0.46): 3. Approval Process - Auto-approval for eligible customers with tenure >24 months. - Manual review by retention team for all others. - Requests must be logged i...

**[4] faq_accounts.md** (similarity 0.40): 8. What are the annual fees and interest charges? Annual membership fees range between ₹500–₹2,000 depending on the card type. The monthly interest rate for unp...

## 5. Task 5: Gradio conversational interface

The UI has a customer ID box, a question box, the chat window, and a side panel showing the customer profile and the sources behind the latest answer.

- **Citations:** each answer carries `[n]` markers, and a "Sources" line with the matching file names is appended under it. The side panel shows the retrieved snippet for each number.
- **Chat history:** past turns are kept in session state and passed back to the chain, so follow-up questions work. "Clear chat" resets it.
- **Missing context:** an unknown customer ID, an off-topic question, an empty question, or an LLM connection error each produce a clear message instead of a crash or a made-up answer.

In [16]:
EXAMPLES = [
    ["How can I waive my annual fee?", ""],
    ["What should we offer to retain customer 15634602?", ""],
    ["Explain the difference between Gold and Platinum cards.", ""],
    ["I'm thinking about closing my account. The fees are too high.", "15737452"],
]


def customer_panel(lookup):
    """Markdown summary of the customer for the side panel."""
    if lookup["status"] == "none":
        return "_No customer selected. Enter an 8-digit customer ID for a personalized answer._"
    if lookup["status"] == "not_found":
        return f"**Customer {lookup['id']} was not found.** Check the ID. Answers will be general."
    c = lookup["profile"]
    return (
        f"**{c['surname']}** (ID {c['customer_id']})\n\n"
        f"- Segment: **{c['retention_segment']}**\n"
        f"- Tenure: {c['tenure_years']} years, credit score {c['credit_score']}\n"
        f"- Products: {c['num_products']}, active member: {'yes' if c['is_active_member'] else 'no'}\n"
        f"- Churn flag: {'yes' if c['exited'] else 'no'}, sentiment: {c['sentiment'] or 'n/a'}\n"
        f"- Risk signals: {'; '.join(c['risk_signals']) or 'none'}"
    )


def build_ui(chain):
    """Task 5: Gradio chat UI with customer ID input, citations, and chat history."""
    import gradio as gr

    def respond(message, customer_id, chat, turns):
        message = (message or "").strip()
        if not message:
            return "", chat, turns, gr.update(), gr.update()
        try:
            result = ask(chain, message, customer_id, turns)
            answer = result["answer"].strip()
            sources = sorted({d.metadata["source"] for d in result["docs"]})
            shown = answer + ("\n\n**Sources:** " + ", ".join(sources) if sources else "")
            panel, cites = customer_panel(result["customer"]), format_citations(result["docs"])
            turns = turns + [(message, answer)]
        except Exception as exc:  # most often: no API key, or Ollama is not running
            shown = ("Sorry, I couldn't reach the language model, so I can't answer right now. "
                     f"Please check the LLM setup and try again. (Details: {type(exc).__name__}: {exc})")
            panel, cites = gr.update(), gr.update()
        chat = chat + [{"role": "user", "content": message}, {"role": "assistant", "content": shown}]
        return "", chat, turns, panel, cites

    def clear():
        return [], [], customer_panel({"status": "none"}), "_Sources for the latest answer appear here._"

    with gr.Blocks(title="Merrick Bank Retention Assistant") as demo:
        gr.Markdown("# Merrick Bank Customer Retention Assistant\n"
                    "Answers are grounded in the bank's policy documents and the customer's churn profile.")
        turns = gr.State([])
        with gr.Row():
            with gr.Column(scale=3):
                try:
                    chatbot = gr.Chatbot(label="Conversation", height=460, type="messages")
                except TypeError:  # Gradio 6 uses the messages format by default and has no "type" argument
                    chatbot = gr.Chatbot(label="Conversation", height=460)
                with gr.Row():
                    customer_id = gr.Textbox(label="Customer ID (optional)", placeholder="e.g. 15634602", scale=1)
                    question = gr.Textbox(label="Question", placeholder="Ask about fees, cards, rewards, retention offers...", scale=3)
                with gr.Row():
                    send = gr.Button("Send", variant="primary")
                    clear_btn = gr.Button("Clear chat")
                gr.Examples(examples=EXAMPLES, inputs=[question, customer_id], label="Sample queries")
            with gr.Column(scale=2):
                gr.Markdown("### Customer")
                profile_md = gr.Markdown(customer_panel({"status": "none"}))
                gr.Markdown("### Sources for the latest answer")
                sources_md = gr.Markdown("_Sources for the latest answer appear here._")

        io = dict(inputs=[question, customer_id, chatbot, turns],
                  outputs=[question, chatbot, turns, profile_md, sources_md])
        send.click(respond, **io)
        question.submit(respond, **io)
        clear_btn.click(clear, outputs=[chatbot, turns, profile_md, sources_md])
    return demo

In [17]:
demo = build_ui(chain)
demo.launch()  # add share=True for a temporary public link

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


The same app can be started outside the notebook with `python app.py`, which serves it at http://127.0.0.1:7860.

## 6. Task 6: Testing and evaluation

The test set is the three sample queries from the assignment plus three that probe the edges: a personalized churn conversation, an unknown customer ID, and an off-topic question.

Two kinds of evaluation run on every answer:

1. **Automated checks** (objective, no LLM involved):
   - `expected_source_retrieved`: did retrieval return the document that actually holds the answer?
   - `citations_valid`: does the answer cite at least one `[n]`, and does every cited number point to a retrieved chunk?
   - `personalized`: when a customer is involved, does the answer mention their name or segment?
   - `handled_missing`: for the edge cases, did the assistant decline or flag the problem instead of inventing an answer?
2. **LLM-as-judge scores** from 1 to 5 for grounding, personalization, tone, and coherence. The judge sees the retrieved context and customer profile, so it can check the answer against them.

In [18]:
TEST_CASES = [
    {"name": "Fee waiver (general)", "question": "How can I waive my annual fee?", "customer_id": "",
     "expected_source": "fee_waiver_policy.txt", "expect_personalized": False, "expect_missing": False},
    {"name": "Retention offer (ID in question)", "question": "What should we offer to retain customer 15634602?", "customer_id": "",
     "expected_source": "retention_policy.txt", "expect_personalized": True, "expect_missing": False},
    {"name": "Gold vs Platinum", "question": "Explain the difference between Gold and Platinum cards.", "customer_id": "",
     "expected_source": "product_summary.txt", "expect_personalized": False, "expect_missing": False},
    {"name": "At-risk customer wants to leave", "question": "I'm thinking about closing my account. The fees are too high.", "customer_id": "15737452",
     "expected_source": None, "expect_personalized": True, "expect_missing": False},
    {"name": "Unknown customer ID", "question": "What retention offer does this customer qualify for?", "customer_id": "99999999",
     "expected_source": "retention_policy.txt", "expect_personalized": False, "expect_missing": True},
    {"name": "Off-topic question", "question": "What is the weather in Columbus today?", "customer_id": "",
     "expected_source": None, "expect_personalized": False, "expect_missing": True},
]

MISSING_PHRASES = ("couldn't find", "could not find", "could not be found", "not found", "don't have", "do not have",
                   "no matching", "unable to find", "check the id", "verify the id", "not able to")

JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are grading a bank customer-retention assistant. Score the ANSWER from 1 (poor) to 5 (excellent) on:\n"
     "- grounding: every factual claim is supported by the CONTEXT or CUSTOMER PROFILE, nothing is invented. "
     "If the context is empty and the answer correctly says it has no information, that is a 5.\n"
     "- personalization: uses the customer profile appropriately. If no customer applies, score how well it avoids assuming one.\n"
     "- tone: empathetic, respectful, plain language, compliant (no promises beyond policy).\n"
     "- coherence: clear, well organized, directly answers the question.\n"
     'Reply with JSON only, like {{"grounding": 5, "personalization": 4, "tone": 5, "coherence": 5, "comment": "one short sentence"}}'),
    ("human", "QUESTION:\n{question}\n\nCUSTOMER PROFILE:\n{customer_block}\n\nCONTEXT:\n{context}\n\nANSWER:\n{answer}"),
])


def extract_citations(answer):
    """Chunk numbers cited in an answer. Accepts [1], [1][2], [1, 2], and [1-3]."""
    cited = []
    for group in re.findall(r"\[([\d\s,\-]+)\]", answer):
        for part in group.split(","):
            part = part.strip()
            if re.fullmatch(r"\d+\s*-\s*\d+", part):
                low, high = (int(n) for n in part.split("-"))
                cited += list(range(low, high + 1))
            elif part.isdigit():
                cited.append(int(part))
    return cited


def automated_checks(case, result):
    """Objective checks on one result."""
    answer, docs, lookup = result["answer"], result["docs"], result["customer"]
    sources = [d.metadata["source"] for d in docs]
    cited = extract_citations(answer)

    checks = {
        "expected_source_retrieved": (case["expected_source"] in sources) if case["expected_source"] else None,
        "citations_valid": (bool(cited) and all(1 <= n <= len(docs) for n in cited)) if docs and not case["expect_missing"] else None,
        "personalized": None,
        "handled_missing": None,
    }
    if case["expect_personalized"] and lookup["status"] == "found":
        profile = lookup["profile"]
        segment_word = profile["retention_segment"].split("-")[0].lower()  # high / moderate / low
        checks["personalized"] = profile["surname"].lower() in answer.lower() or segment_word in answer.lower()
    if case["expect_missing"]:
        checks["handled_missing"] = any(p in answer.lower() for p in MISSING_PHRASES)
    return checks


def judge_answer(llm, result):
    """LLM-as-judge scores for one result. Returns {} if the judge reply cannot be parsed."""
    reply = (JUDGE_PROMPT | llm | StrOutputParser()).invoke({
        "question": result["question"],
        "customer_block": result["customer_block"],
        "context": result["context"],
        "answer": result["answer"],
    })
    match = re.search(r"\{.*\}", reply, flags=re.DOTALL)
    try:
        return json.loads(match.group(0)) if match else {}
    except json.JSONDecodeError:
        return {}


def run_evaluation(chain, llm, cases=TEST_CASES):
    """Run every test case and return (results table, raw results)."""
    rows, raw = [], []
    for case in cases:
        result = ask(chain, case["question"], case["customer_id"])
        scores = judge_answer(llm, result)
        rows.append({
            "test": case["name"],
            "sources": ", ".join(sorted({d.metadata["source"] for d in result["docs"]})) or "(none)",
            **automated_checks(case, result),
            **{k: scores.get(k) for k in ("grounding", "personalization", "tone", "coherence", "comment")},
        })
        raw.append({"case": case, "result": result})
    return pd.DataFrame(rows), raw

In [19]:
eval_df, eval_raw = run_evaluation(chain, llm)

# Each answer is shown as Markdown so long lines wrap and nothing is cut off
for item in eval_raw:
    case, res = item["case"], item["result"]
    retrieved = ", ".join(f"[{i}] {d.metadata['source']} ({d.metadata['similarity']:.2f})"
                          for i, d in enumerate(res["docs"], start=1)) or "(nothing)"
    display(Markdown(
        f"---\n**{case['name']}** | customer ID field: {case['customer_id'] or '(empty)'}\n\n"
        f"**Q: {case['question']}**\n\n{res['answer']}\n\n"
        f"*Retrieved:* {retrieved}\n\n*Citations found in answer:* {extract_citations(res['answer']) or 'none'}"
    ))

---
**Fee waiver (general)** | customer ID field: (empty)

**Q: How can I waive my annual fee?**

To waive your annual fee, you need to meet certain eligibility criteria. You must be an active customer for at least 12 months, have no late payments in the last 3 billing cycles, and either have a credit score above 700 or be experiencing verified financial hardship [1]. 

If you qualify, you can request a one-time annual fee waiver, which can be applied once per card every 12 months [1]. The request will be manually reviewed by the retention team unless you have been a customer for over 24 months, in which case it may be auto-approved [3].

To proceed, you can contact customer support to initiate the request and confirm your eligibility.

*Retrieved:* [1] fee_waiver_policy.txt (0.66), [2] retention_policy.txt (0.51), [3] fee_waiver_policy.txt (0.46), [4] faq_accounts.md (0.40)

*Citations found in answer:* [1, 1, 3]

---
**Retention offer (ID in question)** | customer ID field: (empty)

**Q: What should we offer to retain customer 15634602?**

For customer 15634602, who falls into the moderate-value retention segment, we can offer a 50% annual fee waiver or an EMI interest reduction as retention options. 

Given that the customer has been with us for 2 years and has a credit score of 619, these options align with our retention policy for moderate-value customers [3]. 

Please let me know which option you would like to proceed with, and I can assist you further!

*Retrieved:* [1] retention_policy.txt (0.58), [2] retention_policy.txt (0.48), [3] retention_policy.txt (0.42), [4] retention_policy.txt (0.42)

*Citations found in answer:* [3]

---
**Gold vs Platinum** | customer ID field: (empty)

**Q: Explain the difference between Gold and Platinum cards.**

The Gold and Platinum credit cards from Merrick Bank have several key differences:

1. **Annual Fee**: The Gold card has an annual fee of ₹1,000, while the Platinum card has a higher fee of ₹2,000 [1].

2. **Credit Limit**: The Gold card offers a credit limit ranging from ₹50,000 to ₹2,00,000, whereas the Platinum card has a limit between ₹2,00,000 and ₹5,00,000 [1].

3. **Rewards**: The Gold card provides 2X reward points on groceries, fuel, and dining, while the Platinum card offers a more lucrative 5X points on travel and e-commerce [1][3].

4. **Eligibility**: Both cards require an income above ₹8L per annum and a credit score greater than 750 for eligibility [1].

In summary, the Platinum card is designed for those who spend more on travel and e-commerce, offering higher rewards and a larger credit limit, but at a higher annual fee.

*Retrieved:* [1] product_summary.txt (0.49), [2] rewards_and_benefits.txt (0.40), [3] rewards_and_benefits.txt (0.36), [4] product_summary.txt (0.27)

*Citations found in answer:* [1, 1, 1, 3, 1]

---
**At-risk customer wants to leave** | customer ID field: 15737452

**Q: I'm thinking about closing my account. The fees are too high.**

I understand your concern about the fees. If you decide to close your account, you will need to clear all dues and EMIs first. You can initiate the closure request through the app or at any branch, and you will receive confirmation via your registered email [1].

Before making a final decision, I’d like to mention that as a moderate-value customer, you may be eligible for a 50% annual fee waiver or an EMI interest reduction. This could help reduce your fees significantly. If you're interested, I can assist you with that option. Please let me know how you'd like to proceed!

*Retrieved:* [1] faq_accounts.md (0.52), [2] retention_policy.txt (0.45), [3] faq_accounts.md (0.43), [4] fee_waiver_policy.txt (0.40)

*Citations found in answer:* [1]

---
**Unknown customer ID** | customer ID field: 99999999

**Q: What retention offer does this customer qualify for?**

I couldn't find a matching record for the customer ID provided. Please check the ID and try again. In general terms, retention offers vary based on the customer's profile:

- High-value customers may qualify for a fee waiver up to ₹1,000, bonus reward points, or an interest reduction of 0.5%.
- Moderate-value customers might receive a 50% annual fee waiver or an EMI interest reduction.
- Low-value customers could be offered a free add-on card or personalized financial counseling.

If you have the correct customer ID, please provide it, and I can assist you further.

*Retrieved:* [1] retention_policy.txt (0.69), [2] retention_policy.txt (0.61), [3] retention_policy.txt (0.58), [4] fee_waiver_policy.txt (0.56)

*Citations found in answer:* none

---
**Off-topic question** | customer ID field: (empty)

**Q: What is the weather in Columbus today?**

I'm sorry, I couldn't find anything in Merrick Bank's policy and product documents that answers that, so I don't want to guess. I can help with annual fees and fee waivers, card and account products, rewards, complaints, and retention offers. If you include a customer ID, I can tailor the answer to that customer. For anything else, please contact customer support.

*Retrieved:* (nothing)

*Citations found in answer:* none

In [20]:
pd.set_option("display.max_colwidth", 120)
eval_df.to_csv(BASE_DIR / "evaluation_results.csv", index=False)
eval_df

,test,sources,expected_source_retrieved,citations_valid,personalized,handled_missing,grounding,personalization,tone,coherence,comment
0,Fee waiver (general),"faq_accounts.md, fee_waiver_policy.txt, retention_policy.txt",True,True,None,None,5,5,5,5,The answer is clear and well-structured.
1,Retention offer (ID in question),retention_policy.txt,True,True,True,None,5,5,5,5,The answer is clear and aligns well with the retention policy.
2,Gold vs Platinum,"product_summary.txt, rewards_and_benefits.txt",True,True,None,None,5,5,5,5,The answer clearly explains the differences without assuming customer details.
3,At-risk customer wants to leave,"faq_accounts.md, fee_waiver_policy.txt, retention_policy.txt",None,True,True,None,5,4,5,5,The answer is clear and provides relevant information while addressing the customer's concerns.
4,Unknown customer ID,"fee_waiver_policy.txt, retention_policy.txt",True,None,None,True,5,5,5,5,The answer is clear and appropriately addresses the lack of customer information.
5,Off-topic question,(none),None,None,None,True,5,5,5,5,The answer is clear and appropriately addresses the lack of information.


In [21]:
# Summary: pass rate for each automated check, and the average judge score for each criterion
check_cols = ["expected_source_retrieved", "citations_valid", "personalized", "handled_missing"]
score_cols = ["grounding", "personalization", "tone", "coherence"]

summary = pd.concat([
    eval_df[check_cols].apply(lambda col: f"{int(col.dropna().sum())} of {col.notna().sum()} passed").rename("result"),
    eval_df[score_cols].apply(pd.to_numeric, errors="coerce").mean().round(2).astype(str).add(" / 5 average").rename("result"),
]).to_frame()
summary

,result
expected_source_retrieved,4 of 4 passed
citations_valid,4 of 4 passed
personalized,2 of 2 passed
handled_missing,2 of 2 passed
grounding,5.0 / 5 average
personalization,4.83 / 5 average
tone,5.0 / 5 average
coherence,5.0 / 5 average


In [22]:
# Chat history check: the follow-up only makes sense if the previous turn is remembered
turns = []
first = ask(chain, "Explain the difference between Gold and Platinum cards.", "", turns)
turns.append(("Explain the difference between Gold and Platinum cards.", first["answer"]))
follow_up = ask(chain, "Which of those two has the lower annual fee, and by how much?", "", turns)
display(Markdown(follow_up["answer"]))

The Gold credit card has the lower annual fee of ₹1,000 compared to the Platinum card, which has an annual fee of ₹2,000. The difference in the annual fee is ₹1,000 [1]. If you have any more questions or need further assistance, feel free to ask!

### Evaluation notes

How to read the results above:

- **Grounding accuracy:** `expected_source_retrieved` confirms the retriever found the right document for each sample query (fee waiver policy, retention policy, product summary). `citations_valid` confirms each answer points back to chunks that were actually retrieved. The judge's grounding score checks the wording of the answer against those chunks.
- **Personalization:** for customer 15634602 the profile shows 2 years of tenure and a credit score of 619, which lands in the Moderate-value segment. A correct answer should offer what the retention policy lists for that segment (a 50% annual fee waiver or EMI interest reduction) and nothing from the High-value tier.
- **Tone:** the system prompt carries the bank's own script guidelines (acknowledge the concern, plain language, one relevant option, clear next step), and the judge scores against them.
- **Coherence:** scored by the judge, and the follow-up test confirms the assistant keeps track of the conversation.
- **Missing context:** the unknown customer ID and the off-topic question should both pass `handled_missing`. The off-topic one never reaches the LLM at all, because nothing clears the similarity floor.

**Limitations and next steps**

- The judge is the same LLM that wrote the answers, so its scores are a sanity check and not an independent audit. A stronger setup would use a different model as the judge plus a human spot check.
- Retrieval uses only the current question. A follow-up like "what about the other one?" relies on chat history for meaning and may retrieve weaker chunks. Rewriting follow-ups into standalone questions before retrieval would fix that.
- Late payment history and hardship status are not in the dataset, so fee waiver eligibility can only be partly determined. The assistant is told to say those still need verification.
- The knowledge base is six short documents. With a larger corpus, a reranker and metadata filters by document type would be worth adding.

### Observations from this run

Reading the answers by hand turned up two things the scores did not catch:

- **A grounding error in the Gold vs Platinum answer.** The assistant said both cards require income above ₹8L and a credit score above 750. In `product_summary.txt` that requirement applies to the Platinum card only. The citation was valid in the sense that it pointed to the right chunk, but the claim did not match the chunk, and the LLM judge still scored grounding 5 out of 5. This shows the limits of both checks: the citation check confirms a source was cited, not that it was read correctly, and a judge from the same model family is too lenient.
- **The at-risk customer answer was only partly personalized.** It offered the correct Moderate-value retention option, but it did not cite the retention policy for that offer, it did not use the customer's name, and it told the customer their internal segment label, which a real agent would not say out loud.

Next steps would be a claim-by-claim grounding check (verify each cited sentence against its chunk), a different and stricter model as the judge, and a prompt rule that product details must be attributed only to the product they are listed under.